# 🔬 RAGAS Benchmark — ai-v3 RAG Chatbot

Benchmark đánh giá hệ thống RAG Chatbot bằng RAGAS framework (4 metrics: Faithfulness, Answer Relevancy, Context Recall, Context Precision).

**Runtime:** Chọn T4 GPU (Runtime → Change runtime type → T4)

In [ ]:
#@title 📦 Step 1: Clone repo & setup
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
%cd YOUR_REPO
!git checkout ai
%cd ai-v3

# Install dependencies
!pip install -q ragas>=0.2.0 langchain-google-genai>=2.0.0 datasets>=2.14.0
!pip install -q sentence-transformers tqdm numpy pandas
!pip install -q google-generativeai langchain-community langchain-huggingface
!pip install -q rank-bm25 supabase

print('✅ Dependencies installed!')

In [ ]:
#@title 🔑 Step 2: Setup API Keys
import os

# ⚠️ THAY API KEY CỦA BẠN VÀO ĐÂY
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'  #@param {type:"string"}

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['GEMINI_MODEL'] = 'gemini-3.1-flash-lite'  # 500 quota/ngày

# Supabase DB
os.environ['DB_HOST'] = 'aws-0-ap-southeast-1.pooler.supabase.com'
os.environ['DB_PORT'] = '6543'
os.environ['DB_NAME'] = 'postgres'
os.environ['DB_USER'] = 'postgres.zzukpubwbntihzztilqy'
os.environ['DB_PASSWORD'] = 'agW24oOesftDhJkA'

print(f'✅ API Key set! Model: {os.environ["GEMINI_MODEL"]}')

In [ ]:
#@title ✅ Step 3: Verify setup
import sys
sys.path.insert(0, '.')

# Test DB
from core.db import fetch_all_products
products = fetch_all_products()
print(f'✅ Database: {len(products)} products loaded')

# Test Gemini API
import google.generativeai as genai
genai.configure(api_key=os.environ['GEMINI_API_KEY'])
model = genai.GenerativeModel('gemini-3.1-flash-lite')
resp = model.generate_content('Say hello in Vietnamese')
print(f'✅ Gemini API: {resp.text[:50]}')

In [ ]:
#@title 🧪 Step 4: Test run (2 questions)
import run_ragas_benchmark

# Override to 2 questions for testing
original_func = run_ragas_benchmark.generate_100_eval_dataset

def test_dataset(seed=42):
    import random
    random.seed(seed)
    dataset = []
    for i in range(1, 3):
        intent, tmpl = random.choice(run_ragas_benchmark.TEMPLATES)
        brand = random.choice(run_ragas_benchmark.BRANDS)
        brand2 = random.choice([b for b in run_ragas_benchmark.BRANDS if b != brand])
        ptype = random.choice(run_ragas_benchmark.PRODUCT_TYPES)
        ptype2 = random.choice(run_ragas_benchmark.PRODUCT_TYPES)
        spec = random.choice(run_ragas_benchmark.SPECS_KEYS)
        ucase = random.choice(run_ragas_benchmark.USE_CASES)
        budget = random.choice(run_ragas_benchmark.BUDGETS)
        q = tmpl.format(product_type=ptype, product_type2=ptype2, brand=brand, brand2=brand2, spec_key=spec, use_case=ucase, budget=budget)
        gt = f'Sản phẩm {ptype} của thương hiệu {brand} ({spec}) phù hợp cho {ucase} với mức giá khoảng {budget} triệu VNĐ.'
        dataset.append({'id': i, 'question': q, 'intent': intent, 'expected_brand': brand, 'expected_category': ptype, 'ground_truth': gt})
    return dataset

run_ragas_benchmark.generate_100_eval_dataset = test_dataset
result = run_ragas_benchmark.run_100_ragas_benchmark()
print('\n✅ Test run completed!')

In [ ]:
#@title 🚀 Step 5: Run full 100 questions
# Uncomment to run:

# run_ragas_benchmark.generate_100_eval_dataset = original_func
#
# # Clear checkpoint (optional - remove to resume from last run)
# import os
# cp = 'eval/cache/ragas_100_batch_checkpoint.json'
# if os.path.exists(cp):
#     os.remove(cp)
#     print('🗑️ Checkpoint cleared')
#
# result = run_ragas_benchmark.run_100_ragas_benchmark()
# print('\n🎉 Full benchmark completed!')

In [ ]:
#@title 📥 Step 6: Download results
from google.colab import files

# Download JSON results
files.download('eval/ragas_eval_results.json')

# Download Markdown report
files.download('eval/ragas_benchmark_report.md')

# Download checkpoint (for resume)
files.download('eval/cache/ragas_100_batch_checkpoint.json')

---
## 📊 Xem kết quả nhanh

Sau khi chạy xong, chạy cell dưới để xem bảng điểm:

In [ ]:
#@title 📊 View results summary
import json

with open('eval/ragas_eval_results.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

scores = data['aggregate_scores']
quality = data.get('evaluation_quality', {})
perf = data.get('performance', {})

print('=' * 60)
print('📊 RAGAS BENCHMARK RESULTS')
print('=' * 60)
print(f'\n📈 RAGAS Scores:')
print(f'  Faithfulness:      {scores["faithfulness"]:.4f}')
print(f'  Answer Relevancy:  {scores["answer_relevancy"]:.4f}')
print(f'  Context Recall:    {scores["context_recall"]:.4f}')
print(f'  Context Precision: {scores["context_precision"]:.4f}')
print(f'  OVERALL:           {scores["ragas_overall"]:.4f}')

if quality:
    print(f'\n🔍 Evaluation Quality:')
    print(f'  LLM Judge:  {quality.get("llm_judge_count", "N/A")}')
    print(f'  Fallback:   {quality.get("fallback_count", "N/A")}')
    print(f'  Total:      {quality.get("scored_total", "N/A")}')

if perf:
    print(f'\n⚡ Performance:')
    print(f'  Latency (mean): {perf.get("latency_mean_ms", 0):.2f} ms')
    print(f'  Throughput:     {perf.get("throughput_qps", 0):.2f} queries/sec')
    print(f'  Est. Cost:      ${perf.get("estimated_cost_usd", 0):.4f}')